In [ ]:
import os
import random
import numpy as np
import torch
import optuna
import pandas as pd

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import VecNormalize

from agent import FloodWarningEnv

In [ ]:
SEED = 1
N_TRIALS_PER_RUN = 50
TRAIN_STEPS = 300_000
N_EVAL_EPISODES = 50

STUDY_NAME = "flood_warning_tuning_lstm_vecnorm"
OUTPUT_DIR = "./tuning/tuning_lstm_vecnorm/"
DB_PATH = "sqlite:///./tuning/tuning_lstm_vecnorm/optuna_study_lstm_vecnorm.db"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def make_env():
    return lambda: FloodWarningEnv()

def objective(trial):
    # PPO hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [256, 512, 1024])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    n_epochs = trial.suggest_int("n_epochs", 5, 20)
    ent_coef = trial.suggest_float("ent_coef", 1e-4, 0.1, log=True)

    # Set seeds
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # Training environment with VecNormalize
    env = make_vec_env(make_env(), n_envs=8, seed=SEED)
    env = VecNormalize(env, norm_obs=True, norm_reward=False)

    # Evaluation environment with VecNormalize, training=False to not update stats
    eval_env = make_vec_env(make_env(), n_envs=1, seed=SEED)
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)
    eval_env.obs_rms = env.obs_rms

    # Agent
    model = RecurrentPPO(
        "MlpLstmPolicy",   # LSTM policy, maintains hidden state across timesteps within an episode
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        ent_coef=ent_coef,
        gamma=0.99,
        verbose=0,
        seed=SEED,
    )

    # Evaluation callback
    eval_callback = EvalCallback(
        eval_env,
        eval_freq=25_000,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
        verbose=0,
    )

    model.learn(total_timesteps=TRAIN_STEPS, callback=eval_callback)

    mean_reward = eval_callback.last_mean_reward

    env.close()
    eval_env.close()
    return mean_reward

if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        storage=DB_PATH,
        study_name=STUDY_NAME,
        load_if_exists=True
    )
    study.optimize(objective, n_trials=N_TRIALS_PER_RUN)

    print("Best trial:")
    print(f"  Mean reward: {study.best_trial.value:.4f}")
    print(f"  Params: {study.best_trial.params}")

    study.trials_dataframe().to_csv(os.path.join(OUTPUT_DIR, "study_results.csv"), index=False)
    print("Study results saved to " + os.path.join(OUTPUT_DIR, "study_results.csv"))

In [ ]:
# sensitivity plots

In [ ]:
df = pd.read_csv("tuning/tuning_lstm_vecnorm/study_results.csv")
best_row = df.loc[df["value"].idxmax()]
print(best_row)

In [ ]:
SEED = 1
TRAIN_STEPS = 10_000_000
N_EVAL_EPISODES = 50

OUTPUT_DIR = "./models/lstm_vecnorm/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Hyperparameters ---
LEARNING_RATE = 1.1364553022898512e-05 #0.000252
N_STEPS = 512 #1024
BATCH_SIZE = 64 #32
N_EPOCHS = 9 #6
ENT_COEF = 0.049013358593025094 #0.000897

# --- Reward weights ---
FALSE_WEIGHT =  0.898965734002638 #0.301511
MISSED_WEIGHT = 1.0000745120301147 #1.007163
JUMP_WEIGHT = 0.8346345532448564 #0.878312


def make_env(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT):
    return lambda: FloodWarningEnv(false_weight=false_weight, missed_weight=missed_weight, jump_weight=jump_weight)


# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Training environment with VecNormalize
env = make_vec_env(make_env(), n_envs=8, seed=SEED)
env = VecNormalize(env, norm_obs=True, norm_reward=False)

# Evaluation environment — training=False to not update obs stats
eval_env = make_vec_env(make_env(), n_envs=1, seed=SEED)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)
eval_env.obs_rms = env.obs_rms

# Agent
model = RecurrentPPO(
    "MlpLstmPolicy",
    env,
    learning_rate=LEARNING_RATE,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    ent_coef=ENT_COEF,
    gamma=0.99,
    verbose=1,
    seed=SEED,
)

# Evaluation callback
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=OUTPUT_DIR,
    eval_freq=25_000,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    verbose=1,
)

model.learn(total_timesteps=TRAIN_STEPS, callback=eval_callback)

# Save final model and VecNormalize stats
model.save(os.path.join(OUTPUT_DIR, "final_model"))
env.save(os.path.join(OUTPUT_DIR, "vec_normalize.pkl"))

print(f"Training complete.")
print(f"Best mean reward: {eval_callback.best_mean_reward:.4f}")
print(f"Final mean reward: {eval_callback.last_mean_reward:.4f}")
print(f"Model and stats saved to {OUTPUT_DIR}")

env.close()
eval_env.close()